# Lecture 5: Neural Networks

*Notebook for UMich EECS 498 (Justin Johnson, Fall 2019), Lecture 5, with material from Stanford CS231n 2026, Lecture 4.*

Linear classifiers draw straight lines and learn one template per class. That's not enough for real images. **Neural networks** fix this by stacking layers with a **nonlinearity** in between.

**What you'll learn**
1. Feature transforms: the old way to get past straight lines.
2. What a neural network computes: $f = W_2\max(0, W_1x)$.
3. How to build and train a 2-layer network from scratch.
4. Why the nonlinearity is essential, and the common **activation functions**.
5. Network size vs. regularization, "space warping", and universal approximation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. The problem, and the old solution: feature transforms

Here's data a linear classifier can't handle: one class inside a ring of the other.

In [ ]:
rng = np.random.default_rng(0)
points = rng.uniform(-2, 2, size=(400, 2))
radius = np.sqrt(points[:, 0]**2 + points[:, 1]**2)
ring_labels = (radius < 1.2).astype(int)          # class 1 inside the circle, class 0 outside

plt.figure(figsize=(3.5, 3.5))
plt.scatter(points[:, 0], points[:, 1], c=ring_labels, cmap="coolwarm", s=8)
plt.gca().set_aspect("equal"); plt.title("no straight line separates these")
plt.show()

**Feature transform:** compute new features in which the data *is* separable. Here, the distance from the center (the radius) does it: class 1 has small $r$, class 0 large $r$. A linear classifier on $r$ is just a threshold.

Let's write a small logistic-regression trainer (a linear classifier for two classes) and compare raw coordinates with the radius feature:

In [ ]:
def train_linear_binary(features, labels, steps=2000, lr=0.5):
    # Logistic regression: score = features . w + b, probability = sigmoid(score).
    w = np.zeros(features.shape[1])
    b = 0.0
    for step in range(steps):
        prob = 1 / (1 + np.exp(-(features @ w + b)))
        error = prob - labels                      # gradient of the cross-entropy loss w.r.t. the score
        w -= lr * features.T @ error / len(labels)
        b -= lr * error.mean()
    predictions = (features @ w + b) > 0
    return np.mean(predictions == labels)

print(f"always guessing 'outside':      {np.mean(ring_labels == 0):.0%}")
print(f"linear classifier on (x, y):    {train_linear_binary(points, ring_labels):.0%}")
print(f"linear classifier on the radius: {train_linear_binary(radius.reshape(-1, 1), ring_labels):.0%}")

On raw coordinates, the linear classifier does no better than always guessing "outside". Before 2012, computer vision worked exactly like this: people **hand-designed** image features (color histograms, HOG edge-direction histograms, "bag of visual words") and trained a linear classifier on top. **Neural networks learn the features instead.**

## 2. Neural networks

$$\text{Linear classifier: } f = Wx \qquad\qquad \text{2-layer neural network: } f = W_2\max(0, W_1x)$$

* $W_1$ maps the input to a **hidden layer** of $H$ numbers; $\max(0, \cdot)$ is the **ReLU** nonlinearity; $W_2$ maps the hidden layer to the class scores.
* In practice each layer also has a bias: $f = W_2\max(0, W_1x + b_1) + b_2$.
* A 3-layer network: $f = W_3\max(0, W_2\max(0, W_1x))$, and so on.

These are also called **fully connected networks** or **multi-layer perceptrons (MLPs)**. The **depth** is the number of layers; the **width** is the size of each layer.

### A forward pass by hand
2 inputs, 3 hidden units, 2 classes, one input point:

In [ ]:
x = np.array([1.0, -2.0])
W1 = np.array([[0.5, -1.0, 0.2],
               [0.3, 0.4, -0.6]])         # 2 inputs -> 3 hidden units
b1 = np.array([0.2, 0.0, 0.0])
W2 = np.array([[1.0, -1.0],
               [0.5, 0.5],
               [-0.3, 0.8]])              # 3 hidden units -> 2 class scores
b2 = np.zeros(2)

z = x @ W1 + b1           # hidden layer, before the nonlinearity
h = np.maximum(0, z)      # ReLU: negative values become 0
scores = h @ W2 + b2
print("z = x W1 + b1  =", z)
print("h = max(0, z)  =", h)
print("scores = h W2  =", scores)

(As in Lecture 3, code stores examples as rows, so we write `x @ W1` instead of $W_1x$.) The middle hidden unit was negative, so ReLU zeroed it.

## 3. Building and training a 2-layer network

We'll classify the **spiral** dataset: three interleaved spiral arms, a classic example from CS231n.

In [ ]:
def make_spiral(points_per_class=100, noise=0.2, seed=0):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for k in range(3):
        r = np.linspace(0, 1, points_per_class)
        angle = np.linspace(k * 4, (k + 1) * 4, points_per_class) + rng.normal(0, noise, points_per_class)
        X.append(np.column_stack([r * np.sin(angle), r * np.cos(angle)]))
        y.append(np.full(points_per_class, k))
    return np.vstack(X), np.concatenate(y)

X_spiral, y_spiral = make_spiral()
plt.figure(figsize=(3.5, 3.5))
plt.scatter(X_spiral[:, 0], X_spiral[:, 1], c=y_spiral, cmap="brg", s=8)
plt.gca().set_aspect("equal"); plt.title("spiral data")
plt.show()

**The plan:**
1. **Initialize** the weights randomly (small numbers, scaled by $\sqrt{2/\text{inputs}}$; Lecture 10 explains why).
2. **Forward pass:** compute the scores and the softmax loss.
3. **Backward pass:** compute the gradients with the chain rule (Lecture 6 derives these step by step):
   * $\partial L/\partial \text{scores} = $ probabilities − one-hot labels
   * $\partial L/\partial W_2 = h^\top\,(\partial L/\partial\text{scores})$
   * go back through $W_2$, then through ReLU (gradient passes only where $z > 0$), then $\partial L/\partial W_1 = x^\top(\ldots)$
4. **Update** with gradient descent. Repeat.

In [ ]:
def init_network(n_inputs, n_hidden, n_classes, seed=0):
    rng = np.random.default_rng(seed)
    return {
        "W1": rng.normal(0, np.sqrt(2 / n_inputs), size=(n_inputs, n_hidden)),
        "b1": np.zeros(n_hidden),
        "W2": rng.normal(0, np.sqrt(2 / n_hidden), size=(n_hidden, n_classes)),
        "b2": np.zeros(n_classes),
    }

def predict(net, X):
    h = np.maximum(0, X @ net["W1"] + net["b1"])
    scores = h @ net["W2"] + net["b2"]
    return np.argmax(scores, axis=1)

In [ ]:
def loss_and_gradients(net, X, y, reg=0.001):
    N = len(y)
    # ---- forward pass ----
    z = X @ net["W1"] + net["b1"]
    h = np.maximum(0, z)
    scores = h @ net["W2"] + net["b2"]
    scores = scores - scores.max(axis=1, keepdims=True)
    probs = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)
    loss = -np.mean(np.log(probs[np.arange(N), y]))
    loss += reg * (np.sum(net["W1"]**2) + np.sum(net["W2"]**2))
    # ---- backward pass ----
    d_scores = probs.copy()
    d_scores[np.arange(N), y] -= 1          # probabilities minus one-hot labels
    d_scores /= N
    grads = {}
    grads["W2"] = h.T @ d_scores + 2 * reg * net["W2"]
    grads["b2"] = d_scores.sum(axis=0)
    d_h = d_scores @ net["W2"].T            # back through the second layer
    d_z = d_h * (z > 0)                     # back through ReLU
    grads["W1"] = X.T @ d_z + 2 * reg * net["W1"]
    grads["b1"] = d_z.sum(axis=0)
    return loss, grads

**Always gradient-check** (Lecture 4) before trusting the backward pass:

In [ ]:
net = init_network(2, 10, 3, seed=1)
_, grads = loss_and_gradients(net, X_spiral, y_spiral)

h = 1e-5
for name in ["W1", "b1", "W2", "b2"]:
    index = (0, 0) if net[name].ndim == 2 else (0,)       # check one entry of each parameter
    original = net[name][index]
    net[name][index] = original + h; loss_plus, _ = loss_and_gradients(net, X_spiral, y_spiral)
    net[name][index] = original - h; loss_minus, _ = loss_and_gradients(net, X_spiral, y_spiral)
    net[name][index] = original
    numerical = (loss_plus - loss_minus) / (2 * h)
    print(f"{name}: analytic {grads[name][index]:+.6f}, numerical {numerical:+.6f}")

They match, so the gradients are right. Now train:

In [ ]:
def train(net, X, y, learning_rate=1.0, reg=0.001, steps=5000):
    for step in range(steps):
        loss, grads = loss_and_gradients(net, X, y, reg)
        for name in net:
            net[name] -= learning_rate * grads[name]
    return net

net = train(init_network(2, 100, 3), X_spiral, y_spiral)
print(f"2-layer network (100 hidden units): training accuracy {np.mean(predict(net, X_spiral) == y_spiral):.1%}")

Let's draw the decision regions, and compare with a linear classifier (a "network" with no hidden layer):

In [ ]:
grid_x, grid_y = np.meshgrid(np.linspace(-1.2, 1.2, 200), np.linspace(-1.2, 1.2, 200))
grid = np.column_stack([grid_x.ravel(), grid_y.ravel()])

def plot_regions(ax, predict_fn, title):
    regions = predict_fn(grid).reshape(grid_x.shape)
    ax.contourf(grid_x, grid_y, regions, alpha=0.3, levels=[-0.5, 0.5, 1.5, 2.5], cmap="brg")
    ax.scatter(X_spiral[:, 0], X_spiral[:, 1], c=y_spiral, cmap="brg", s=6)
    ax.set_title(title); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])

# a linear classifier: just one weight matrix
rng = np.random.default_rng(0)
W_lin, b_lin = np.zeros((2, 3)), np.zeros(3)
for step in range(2000):
    s = X_spiral @ W_lin + b_lin
    p = np.exp(s - s.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
    p[np.arange(len(y_spiral)), y_spiral] -= 1
    W_lin -= 0.5 * X_spiral.T @ p / len(y_spiral)
    b_lin -= 0.5 * p.mean(axis=0)
linear_predict = lambda X: np.argmax(X @ W_lin + b_lin, axis=1)
print(f"linear classifier: training accuracy {np.mean(linear_predict(X_spiral) == y_spiral):.1%}")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
plot_regions(axes[0], linear_predict, "linear classifier")
plot_regions(axes[1], lambda X: predict(net, X), "2-layer neural network")
plt.show()

**What we see:** the linear classifier can only cut the plane with straight lines. The network bends its boundaries to follow the spiral arms.

### What does the hidden layer learn?
The first layer $W_1$ is a **bank of templates** (here 100 of them), and the second layer **combines** them into class scores. So one class can use several templates. Remember Lecture 3's mirrored digits, where one template per class had to blend a shape with its mirror image? A hidden layer can dedicate different templates to each orientation.

## 4. Why we need the nonlinearity

**Q: What if we leave out the ReLU?** Then

$$f = W_2(W_1x) = (W_2W_1)x = W_3x$$

which is just a linear classifier again! Two linear layers in a row collapse into one. Let's check: train the same network without ReLU.

In [ ]:
def predict_no_relu(net, X):
    h = X @ net["W1"] + net["b1"]                  # no max(0, ...)
    return np.argmax(h @ net["W2"] + net["b2"], axis=1)

net_no_relu = init_network(2, 100, 3)
for step in range(3000):
    z = X_spiral @ net_no_relu["W1"] + net_no_relu["b1"]
    s = z @ net_no_relu["W2"] + net_no_relu["b2"]
    p = np.exp(s - s.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
    p[np.arange(len(y_spiral)), y_spiral] -= 1; p /= len(y_spiral)
    d_z = p @ net_no_relu["W2"].T                  # no ReLU to pass through
    net_no_relu["W2"] -= 0.1 * z.T @ p; net_no_relu["b2"] -= 0.1 * p.sum(0)
    net_no_relu["W1"] -= 0.1 * X_spiral.T @ d_z; net_no_relu["b1"] -= 0.1 * d_z.sum(0)

print(f"network WITHOUT ReLU: {np.mean(predict_no_relu(net_no_relu, X_spiral) == y_spiral):.1%}  (about the same as the linear classifier)")
print(f"network WITH ReLU:    {np.mean(predict(net, X_spiral) == y_spiral):.1%}")

### Activation functions
| Function | Formula | Notes |
|---|---|---|
| **Sigmoid** | $\sigma(x) = 1/(1 + e^{-x})$ | Squashes to (0, 1). **Saturates:** for large $\lvert x\rvert$ its slope is ~0, which kills gradients. |
| **tanh** | $\tanh(x)$ | Squashes to (−1, 1); zero-centered, but still saturates. |
| **ReLU** | $\max(0, x)$ | **The default.** Cheap, doesn't saturate for $x > 0$, trains much faster. Units can "die" if their input is always negative. |
| **Leaky ReLU** | $\max(0.01x, x)$ | Never dies. |
| **GELU** | $x\cdot\Phi(x)$ | Smooth version of ReLU; the default in Transformers. |

In [ ]:
xs = np.linspace(-4, 4, 200)
sigmoid = 1 / (1 + np.exp(-xs))
activations = {"sigmoid": sigmoid, "tanh": np.tanh(xs), "ReLU": np.maximum(0, xs), "Leaky ReLU": np.maximum(0.1 * xs, xs)}

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for name, values in activations.items():
    axes[0].plot(xs, values, label=name)
    axes[1].plot(xs, np.gradient(values, xs), label=name)       # slope of each function
axes[0].set_title("activation functions"); axes[0].legend(); axes[0].set_ylim(-1.5, 4)
axes[1].set_title("their slopes (what backprop multiplies by)")
plt.show()

slope = sigmoid * (1 - sigmoid)
print(f"sigmoid slope: at x = 0: {slope[100]:.2f},  at x = 4: {slope[-1]:.3f}")
print(f"through 10 sigmoid layers, the gradient shrinks by at least 0.25^10 = {0.25**10:.1e}")

The sigmoid's slope is at most 0.25, and almost 0 far from the center. Backprop multiplies by this slope at every layer, so gradients in deep sigmoid networks **vanish**. ReLU's slope is exactly 1 for positive inputs.

## 5. How big should the network be?

More hidden units = more complex boundaries. Let's try different sizes on a **noisier** spiral, and measure accuracy on a separate test spiral:

In [ ]:
X_noisy, y_noisy = make_spiral(noise=0.45, seed=3)
X_test_sp, y_test_sp = make_spiral(noise=0.45, seed=4)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
for ax, n_hidden in zip(axes, [3, 10, 50, 300]):
    net_size = train(init_network(2, n_hidden, 3, seed=1), X_noisy, y_noisy, reg=1e-4, steps=4000)
    train_acc = np.mean(predict(net_size, X_noisy) == y_noisy)
    test_acc = np.mean(predict(net_size, X_test_sp) == y_test_sp)
    print(f"{n_hidden:3d} hidden units: train {train_acc:.0%}, test {test_acc:.0%}")
    regions = predict(net_size, grid).reshape(grid_x.shape)
    ax.contourf(grid_x, grid_y, regions, alpha=0.3, levels=[-0.5, 0.5, 1.5, 2.5], cmap="brg")
    ax.scatter(X_noisy[:, 0], X_noisy[:, 1], c=y_noisy, cmap="brg", s=5)
    ax.set_title(f"{n_hidden} hidden units\ntrain {train_acc:.0%}, test {test_acc:.0%}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

**What we see:** tiny networks can't follow the spiral; bigger ones draw more detailed boundaries. **The lecture's advice: don't make the network small to avoid overfitting. Use a big network and stronger regularization instead.** Here's the 300-unit network with different regularization strengths:

In [ ]:
for reg in [1e-4, 1e-3, 1e-2, 1e-1]:
    net_reg = train(init_network(2, 300, 3, seed=1), X_noisy, y_noisy, reg=reg, steps=4000)
    print(f"lambda {reg:<7g}: train {np.mean(predict(net_reg, X_noisy) == y_noisy):.0%}, "
          f"test {np.mean(predict(net_reg, X_test_sp) == y_test_sp):.0%}")

A small $\lambda$ works well; too much regularization smooths the boundary so much that it can't fit the spiral at all. $\lambda$ is a hyperparameter you tune on validation data.

## 6. Space warping

Why does a hidden layer help? A ReLU layer **folds space**: wherever a unit's input is negative, it's flattened to 0. Points that couldn't be separated by a line before can become separable after the fold, and the final layer draws its straight line *there*.

A network with just **2 hidden units** can classify "the top-right quadrant vs. everything else", which no single line can do. Let's look at the data before and after the hidden layer:

In [ ]:
rng = np.random.default_rng(5)
X_quad = rng.uniform(-1, 1, size=(400, 2))
y_quad = ((X_quad[:, 0] > 0) & (X_quad[:, 1] > 0)).astype(int)   # top-right quadrant = class 1

net_2 = None
for seed in range(10):                                            # try a few random starts, keep the best
    candidate = train(init_network(2, 2, 2, seed=seed), X_quad, y_quad, reg=0, steps=3000)
    if net_2 is None or np.mean(predict(candidate, X_quad) == y_quad) > np.mean(predict(net_2, X_quad) == y_quad):
        net_2 = candidate

hidden = np.maximum(0, X_quad @ net_2["W1"] + net_2["b1"])        # the data after the hidden layer
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].scatter(X_quad[:, 0], X_quad[:, 1], c=y_quad, cmap="coolwarm", s=8); axes[0].set_title("input space")
axes[1].scatter(hidden[:, 0], hidden[:, 1], c=y_quad, cmap="coolwarm", s=8); axes[1].set_title("after the ReLU layer (folded)")
plt.show()
print(f"accuracy with only 2 hidden units: {np.mean(predict(net_2, X_quad) == y_quad):.0%}")

In the folded space, a straight line separates the two colors. More hidden units mean more folds, and more complicated shapes.

## 7. Universal approximation

**A network with one hidden layer can approximate any continuous function, as closely as you like, if it has enough hidden units.**

The intuition from the lecture: one ReLU, $\max(0, x - s)$, is a "hinge" at position $s$. **Four hinges make a bump** (up, flat, down). And **adding up bumps** can draw any curve.

In [ ]:
def relu(x):
    return np.maximum(0, x)

def bump(x, start, top_start, top_end, end):
    # 0 before `start`, rises to 1 at `top_start`, flat until `top_end`, falls to 0 at `end`. Built from 4 ReLUs.
    rise = top_start - start
    fall = end - top_end
    return relu(x - start) / rise - relu(x - top_start) / rise - relu(x - top_end) / fall + relu(x - end) / fall

xs = np.linspace(0, 1, 500)
target = np.sin(2 * np.pi * xs) + xs

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
axes[0].plot(xs, bump(xs, 0.3, 0.4, 0.6, 0.7)); axes[0].set_title("one bump = 4 ReLUs")
for ax, n_bumps in zip(axes[1:], [5, 30]):
    width = 1 / n_bumps
    approximation = np.zeros_like(xs)
    for i in range(n_bumps):
        center = (i + 0.5) * width
        height = np.sin(2 * np.pi * center) + center
        approximation += height * bump(xs, center - width / 2 - 1e-4, center - width / 2 + 1e-4,
                                       center + width / 2 - 1e-4, center + width / 2 + 1e-4)
    ax.plot(xs, target, "k", label="target")
    ax.plot(xs, approximation, "r", label=f"{4 * n_bumps} ReLUs")
    ax.set_title(f"{n_bumps} bumps"); ax.legend(fontsize=8)
plt.show()

More bumps, better approximation. **But be careful:** this says a network *can represent* any function. It doesn't say gradient descent will *find* those weights, or how much data you'd need. (Reminder: kNN is also a "universal approximator".)

## 8. Convex vs. non-convex

A function is **convex** if it's shaped like a bowl: any straight line between two points on the graph stays above it. Convex functions are easy to optimize: gradient descent always reaches the global minimum. **Linear classifiers have convex losses. Neural networks don't.** Look at 1-D slices of each loss along random directions:

In [ ]:
def linear_loss(w_flat):
    W = w_flat.reshape(2, 3)
    s = X_spiral @ W; s = s - s.max(1, keepdims=True)
    p = np.exp(s) / np.exp(s).sum(1, keepdims=True)
    return -np.mean(np.log(p[np.arange(len(y_spiral)), y_spiral]))

def network_loss(w_flat):
    net_flat = {"W1": w_flat[:20].reshape(2, 10), "b1": w_flat[20:30], "W2": w_flat[30:60].reshape(10, 3), "b2": w_flat[60:63]}
    loss, _ = loss_and_gradients(net_flat, X_spiral, y_spiral, reg=0)
    return loss

rng = np.random.default_rng(1)
steps_along = np.linspace(-3, 3, 100)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, loss_fn, size, title in [(axes[0], linear_loss, 6, "linear classifier"), (axes[1], network_loss, 63, "2-layer network")]:
    start = rng.normal(size=size)
    for _ in range(4):
        direction = rng.normal(size=size)
        ax.plot(steps_along, [loss_fn(start + t * direction) for t in steps_along])
    ax.set_title(f"{title}: 1-D slices of the loss"); ax.set_xlabel("step along a random direction")
plt.show()

The linear classifier's slices are smooth bowls; the network's have bumps and kinks. Neural network training is **non-convex** optimization with few guarantees, and yet, in practice, it works remarkably well.

## 9. A word about brains
A biological neuron receives signals through **dendrites**, sums them in the **cell body**, and fires down its **axon** if the total is high enough. An artificial neuron (weighted sum + activation) is loosely inspired by this. **But be careful with the analogy:** real neurons come in many types, dendrites do complex nonlinear computation, and brains aren't organized in neat layers.

## Summary
* **Feature transforms** can make data linearly separable, but hand-designing them is hard.
* A **neural network** learns the features: $f = W_2\max(0, W_1x)$.
* **Training:** forward pass, loss, backward pass (chain rule), gradient descent. Always gradient-check.
* **The nonlinearity is essential:** without it, layers collapse into one linear function.
* **ReLU** is the default activation; sigmoid and tanh saturate.
* Use a **big network + regularization**, not a small network.
* Hidden layers **fold space**; one hidden layer can approximate any function (in principle).

### Check your understanding
1. What happens to a 2-layer network if you remove the ReLU?
2. Why do deep sigmoid networks have trouble learning?
3. Should you use a smaller network to prevent overfitting?
4. Does "universal approximation" mean neural networks can learn anything?

<details><summary>Answers</summary>

1. $W_2(W_1x) = (W_2W_1)x$: it becomes a linear classifier.
2. The sigmoid's slope is at most 0.25 (and ~0 when saturated), so the gradient shrinks at every layer.
3. Generally no: use a big network and tune the regularization strength.
4. No: it says the right weights *exist*, not that training will find them or how much data is needed.
</details>

**Next:** Lecture 6, backpropagation (where those gradient formulas come from).